In [ ]:
import json
import time
from grupo10.utils.logger import get_logger
from grupo10.utils.audit import write_audit_record
from grupo10.ingest import run_incremental_ingestion
from databricks.sdk.runtime import spark, dbutils


In [ ]:
#Instancia logger e criar as variáveis que serão utilizadas em outras células 
logger = get_logger("sqlserver_ingest_single")
AUDIT_TABLE = "lh_nautical.bronze._ingestion_audit"
CONTROL_TABLE = "lh_nautical.bronze._ingestion_control"

DB_USER = dbutils.secrets.get(scope="sqlserver", key="username")
DB_PASSWORD = dbutils.secrets.get(scope="sqlserver", key="password")
DB_HOST = dbutils.secrets.get(scope="sqlserver", key="host")
DB_PORT = dbutils.secrets.get(scope="sqlserver", key="port")
DB_DATABASE = dbutils.secrets.get(scope="sqlserver", key="db")

jdbc_url = f"jdbc:sqlserver://{DB_HOST}:{DB_PORT};databaseName={DB_DATABASE}"

#Disponibiliza para este notebook a política de segurança que foi fornecida pelo task anterior
politica_governanca = dbutils.jobs.taskValues.get(
taskKey="governance_setup", key="politica_governanca", default={}
)


In [ ]:

dbutils.widgets.text("table_config", "")

table_config = json.loads(dbutils.widgets.get("table_config"))
table = table_config["name"]
primary_key = table_config["primary_key"]

start = time.time()
try:
    logger.info(f"Starting ingestion: {table} | primary_key={primary_key}")

    row_count = run_incremental_ingestion(
        spark=spark,
        table_name=table,
        url=jdbc_url,
        user=DB_USER,
        password=DB_PASSWORD,
        control_table=CONTROL_TABLE,
        primary_key=primary_key,
        lines_per_batch=1000,
    )
    
    target_catalog_table = f"lh_nautical.bronze.{table}"
    
    #Adiciona mascaramento nas colunas necessárias
    if table in politica_governanca:
        # Captura as colunas reais da tabela para cruzar com o JSON
        df_target_cols = spark.table(target_catalog_table).columns
        
        for coluna_sensivel in politica_governanca[table]:
            if coluna_sensivel in df_target_cols:
                logger.info(f"Applying Unity Catalog Mask to: {target_catalog_table}.{coluna_sensivel}")
                spark.sql(f"""
                    ALTER TABLE {target_catalog_table}
                    ALTER COLUMN {coluna_sensivel} SET MASK lh_nautical.bronze.regra_mascaramento
                """)    

    duration = time.time() - start
    logger.info(f"Done: {table} | rows~{row_count} | duration={duration:.2f}s")

    #Registra na tabela de auditoria
    write_audit_record(
        spark=spark,
        table_name=table,
        status="success",
        duration_seconds=duration,
        row_count=row_count,
        audit_table=AUDIT_TABLE,
    )

#Da raise no caso de exception
except Exception as e:
    duration = time.time() - start
    logger.error(f"Failed: {table} | duration={duration:.2f}s | error={e}")

    write_audit_record(
        spark=spark,
        table_name=table,
        status="failed",
        duration_seconds=duration,
        error_message=str(e),
        audit_table=AUDIT_TABLE,
    )
    raise
